# Conformal prediction — scores, sets, and the weighted restore

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/f-inverse/jammi-ai/blob/py-v0.49.1/cookbook/notebooks/book/08-conformal/conformal.ipynb)

Built from [`cookbook/book/chapters/08-conformal/conformal.qmd`](https://github.com/f-inverse/jammi-ai/blob/main/cookbook/book/chapters/08-conformal/conformal.qmd). Run the setup
cell first; every other cell runs top to bottom.

In [ ]:
# Setup: jammi 0.49.1 — the CUDA engine on an sm_80+ GPU (L4, A100, …), the
# CPU engine otherwise — and the cookbook's library and fixtures. On that GPU the
# chapter runs at `full` scale, over the published data; set SCALE = "small" to
# run the seconds-long version over the committed fixtures instead.
import os
import subprocess
import sys


def compute_capability() -> float:
    try:
        out = subprocess.run(
            ["nvidia-smi", "--query-gpu=compute_cap", "--format=csv,noheader"],
            capture_output=True, text=True, check=True,
        ).stdout.split()
    except (OSError, subprocess.CalledProcessError):
        return 0.0
    return float(out[0]) if out else 0.0


gpu = compute_capability() >= 8.0
engine = "jammi-ai-native-cu12" if gpu else "jammi-ai-native"
server = "jammi-server-cu12" if gpu else "jammi-server"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "jammi-ai==0.49.1", engine + "==0.49.1", "jammi-cookbook==0.49.1"], check=True)
SCALE = "full" if gpu else "small"
os.environ["JAMMI_COOKBOOK_SCALE"] = SCALE
print(f"engine: {engine}   scale: {SCALE}")

In [ ]:
import jammi_cookbook

**Recipe:** `conformalize` (`score ∈ {lac, aps, raps}`) · `conformalize_interval` ·
`conformalize_cqr` · **Theory:** distribution-free finite-sample coverage under
exchangeability (Vovk et al. 2005; Angelopoulos & Bates 2021), the nonconformity-score families
LAC / APS / RAPS (Romano et al. 2020; Angelopoulos et al. 2021) and CQR (Romano et al. 2019), and
the covariate-shift repair of weighted split-conformal (Tibshirani et al. 2019; Barber et al. 2023) · **Rails:** measurement (coverage, set-size, width).

This is a study of the engine's **conformal numerics** on tier 04's live
predictions. It carries two payloads:

1. **A measured tour of the score families** — APS, RAPS, LAC for the
   subject-classification sets; the absolute-residual interval and CQR for the
   year-regression intervals. Every coverage and size is measured here and
   checked against a frozen golden.
2. **The score-aligned weighted restore** — the keystone (tier 04) showed that on
   the *real* ogbn-arxiv time-split, importance-weighting is a **no-op** because the
   shift is nearly orthogonal to the conformal score. Here we demonstrate the
   **complement**: a *transparently-synthetic, clearly-labelled* covariate shift that
   **moves the score distribution**, on which weighted split-conformal (Tibshirani et al. 2019)
   **genuinely restores** coverage to nominal. The two results are the two halves of one
   thesis: **weighted conformal repairs a shift if and only if the shift moves the
   nonconformity-score distribution.**

First, tier 04's substrate — the subject classifier's scores and the year
predictor's served distributions over the calibration (2018) and test (2019–)
eras:

In [ ]:
import tempfile

import jammi
import numpy as np
from jammi_cookbook import contracts, datasets, keystone, scale, shift

SCALE = scale.current()
alpha = 0.10
db = jammi.connect(f"file://{tempfile.mkdtemp()}")
arxiv = datasets.arxiv(db, SCALE)
embeddings = keystone.embed(db, arxiv, SCALE)
propagated = keystone.propagate(db, arxiv, embeddings)
scores = keystone.subject_scores(db, arxiv, propagated)
predictor = keystone.train_year_predictor(db, arxiv, SCALE, propagated)
print(f"target coverage 1 − α = {1 - alpha:.2f}")

## Part A — the classification scores: LAC, APS, RAPS

The classification task is subject prediction; the predictor is the tier-04
nearest-neighbour vote on the **propagated** (citation-graph conditioned)
embeddings — a paper's class scores are the similarity-weighted subjects of its
nearest training-era papers. We form prediction *sets* with each nonconformity
family.

In [ ]:
cal_scores, cal_labels = scores.cal_scores.tolist(), scores.cal_labels.tolist()
test_scores, test_labels = scores.test_scores.tolist(), scores.test_labels.tolist()
print(f"calibration rows: {len(cal_labels)}   test rows: {len(test_labels)}   "
      f"classes: {len(scores.classes)}")

The three families differ only in the nonconformity they rank by:

* **LAC** (least-ambiguous set classifier) — score is $1 - \hat p_y$; the smallest
  sets at *exact* nominal calibration but the least adaptive (Romano et al. 2020).
* **APS** (adaptive prediction sets) — score is the cumulative softmax mass up to the
  true class; adaptive, with conditional-coverage behaviour (Romano et al. 2020).
* **RAPS** (regularized APS) — APS with a rank-penalty $(\lambda, k_{\text{reg}})$ that
  discourages long tails (Angelopoulos et al. 2021).

In [ ]:
def coverage_size(sets):
    cov = float(np.mean([y in s for y, s in zip(test_labels, sets)]))
    size = float(np.mean([len(s) for s in sets]))
    return cov, size


lac_sets = db.conformalize(cal_scores, cal_labels, test_scores, alpha=alpha, score="lac")
aps_sets = db.conformalize(cal_scores, cal_labels, test_scores, alpha=alpha, score="aps")
# RAPS takes raps_params = (lambda, k_reg): the penalty weight and the 1-based rank
# past which it bites.
raps_sets = db.conformalize(
    cal_scores, cal_labels, test_scores, alpha=alpha, score="raps", raps_params=(0.1, 1)
)

lac_cov, lac_size = coverage_size(lac_sets)
aps_cov, aps_size = coverage_size(aps_sets)
raps_cov, raps_size = coverage_size(raps_sets)
print(f"LAC : coverage {lac_cov:.3f}   mean set size {lac_size:.2f}")
print(f"APS : coverage {aps_cov:.3f}   mean set size {aps_size:.2f}")
print(f"RAPS: coverage {raps_cov:.3f}   mean set size {raps_size:.2f}")

In [ ]:
contracts.assert_close("arxiv.conformal.lac_coverage", lac_cov, tol=0.03)
contracts.assert_close("arxiv.conformal.lac_set_size", lac_size, tol=0.6)
contracts.assert_close("arxiv.conformal.aps_coverage", aps_cov, tol=0.03)
contracts.assert_close("arxiv.conformal.aps_set_size", aps_size, tol=0.6)
contracts.assert_close("arxiv.conformal.raps_coverage", raps_cov, tol=0.03)
contracts.assert_close("arxiv.conformal.raps_set_size", raps_size, tol=0.6)
if SCALE is scale.Scale.FULL:
    # Under the time split every family under-covers; APS is the sharper family
    # and LAC buys coverage with size.
    assert lac_cov < 1 - alpha and aps_cov < 1 - alpha
    assert aps_size < lac_size and lac_cov > aps_cov

Two facts, measured at `full` scale. First, **every family under-covers** the nominal 0.90: the
later-era test papers are not exchangeable with the earlier-era calibration papers (a
covariate shift carried along the homophilous citation graph), so the marginal
quantile is mis-calibrated regardless of which score we rank by (Barber et al. 2023). The
under-coverage is a property of the *shift*, not the score family.

Second, the size ordering is the honest, *measured* one — not the idealized
textbook ordering. **APS gives the sharper sets** and **LAC the larger ones**,
with the higher realised coverage. The
textbook result that LAC yields the smallest sets holds at *exact* nominal
calibration; under this time-split LAC's $1-\hat p$ threshold lands more
conservatively, admitting more near-ties, so it buys coverage with size while APS
stays sharp but under-covers more.

**RAPS can reduce to APS.** At `full` scale, with this class count and these
calibrated scores, the rank penalty does not change the admitted sets — RAPS
coverage and size equal APS to the digit. The regularization bites only when
sets are long relative to the penalty rank; here the APS sets are already short
enough that the penalty never excludes a class the unregularized APS admits.
The cell above prints both, so you can see whether it bites at the scale you
ran.

## Part B — the regression intervals: absolute-residual and CQR

The regression task is paper-year prediction; the Gaussian context predictor
serves a mean and a standard deviation per paper. Two interval constructions:

* **Absolute-residual** (`conformalize_interval`) — calibrate $|y - \hat\mu|$ on the
  calibration era, apply the quantile symmetrically. A *constant-width* band.
* **CQR** (`conformalize_cqr`) — calibrate the conformity of the predictor's own
  $[\hat\mu - \hat\sigma,\ \hat\mu + \hat\sigma]$ band, so the interval inherits the
  predictor's *heteroscedastic* width (Romano et al. 2019).

In [ ]:
year = {
    r["paper_id"]: r["year"]
    for r in db.sql(f"SELECT paper_id, year FROM {arxiv.papers}.public.{arxiv.papers}").to_pylist()
}
cal_ids, test_ids = arxiv.split["valid"], arxiv.split["test"]
cal_mean, cal_sd = (a.tolist() for a in keystone.predict_years(db, arxiv, predictor, cal_ids))
test_mean, test_sd = (a.tolist() for a in keystone.predict_years(db, arxiv, predictor, test_ids))
cal_year = [float(year[k]) for k in cal_ids]
test_year = [year[k] for k in test_ids]

# Absolute-residual interval.
iv = db.conformalize_interval(cal_mean, cal_year, test_mean, alpha=alpha)
iv_cov = float(np.mean([lo <= test_year[i] <= hi for i, (lo, hi) in enumerate(iv)]))
iv_width = float(np.mean([hi - lo for lo, hi in iv]))

# CQR over the predictor's ±1σ band (the band CQR conformalizes).
cal_lo = [m - s for m, s in zip(cal_mean, cal_sd)]
cal_hi = [m + s for m, s in zip(cal_mean, cal_sd)]
test_lo = [m - s for m, s in zip(test_mean, test_sd)]
test_hi = [m + s for m, s in zip(test_mean, test_sd)]
cqr = db.conformalize_cqr(cal_lo, cal_hi, cal_year, test_lo, test_hi, alpha=alpha)
cqr_cov = float(np.mean([lo <= test_year[i] <= hi for i, (lo, hi) in enumerate(cqr)]))
cqr_width = float(np.mean([hi - lo for lo, hi in cqr]))

print(f"abs-residual interval: coverage {iv_cov:.3f}   width {iv_width:.2f} years")
print(f"CQR interval:          coverage {cqr_cov:.3f}   width {cqr_width:.2f} years")

In [ ]:
contracts.assert_close("arxiv.conformal.interval_coverage", iv_cov, tol=0.04)
contracts.assert_close("arxiv.conformal.interval_width", iv_width, tol=0.6)
contracts.assert_close("arxiv.conformal.cqr_coverage", cqr_cov, tol=0.04)
contracts.assert_close("arxiv.conformal.cqr_width", cqr_width, tol=0.6)
if SCALE is scale.Scale.FULL:
    # Both under-cover under the time split; CQR is wider (it inherits the
    # predictor's spread) and recovers a little more coverage.
    assert iv_cov < 1 - alpha and cqr_cov < 1 - alpha
    assert cqr_width > iv_width and cqr_cov > iv_cov

At `full` scale both intervals **under-cover** under the time split — the same non-exchangeability
lesson as the classification sets. CQR's interval is **wider** (it inherits the
predictor's $\hat\sigma$ rather than imposing one constant half-width) and recovers a
little more coverage, but not to nominal. The keystone (tier 04) carries the *why*:
the predictor's point prediction is essentially unbiased across eras, but the test
era's residuals genuinely run larger — yet residual size is uncorrelated with
test-likeness *within* the calibration set, so no amount of importance-reweighting
on the calibration era can see (or repair) that bias. Weighting that interval is a
no-op there.

## Part C — the score-aligned weighted restore (the keystone's complement)

Here is the genuinely novel payoff of this chapter. The keystone showed that on the
**real** ogbn-arxiv time-split, importance-weighted conformal (Tibshirani et al. 2019) is a
**no-op**, with the diagnostic that the shift is nearly **orthogonal** to the
conformal score: `corr(nonconformity, test-likeness)` is near zero. The lesson there was
the *negative* one — weighting cannot repair a shift the score cannot see.

This is the **complement**, and we are explicit about its status:

::: {.callout-warning}
## This shift is a constructed teaching device

The covariate shift in this section is **transparently synthetic and deliberately
constructed**. It is **not** a property of the real ogbn-arxiv time-split, and we make
**no** claim that the real time-split is repairable by weighting — it is not (the
keystone shows the orthogonal-shift no-op, and the regression case is a location
shift). The purpose here is to exhibit the *other half* of the theorem: the case where
the shift **does** move the nonconformity-score distribution, so that the weighted
restore is real. We label it as a device precisely so the two halves stay honest.
:::

The construction. We take only the genuinely-exchangeable **calibration-era pool** and
split it 50/50 into a synthetic test holdout and a calibration pool — so the *base*
data is exchangeable and the **only** shift is the one we inject. We then bias the
calibration subsample along a **real covariate**: a nearest-neighbour regression of
the APS nonconformity on the paper's neighbourhood — the mean nonconformity of its
nearest calibration-era papers, one query-by-example `search` each. It is a genuine
feature of each paper's position in the embedding space that *correlates with the
score* (by construction of the regression), so over-sampling its low end pulls the
calibration toward **easy** (low-nonconformity) points — a shift that **moves the
score distribution**. We use one
consistent APS nonconformity throughout, so the marginal and weighted passes differ
only in the weights.

In [ ]:
nc = shift.aps_nonconformity(scores.cal_scores, scores.cal_labels)

# A REAL covariate axis along which the score varies: each paper's neighbours' mean
# nonconformity. (This is the transparently-synthetic part — we regress onto the
# score, on purpose.)
nc_of = dict(zip(scores.cal_keys, nc))
shift_axis = np.array([
    np.mean([nc_of[n["paper_id"]] for n in keystone.neighbours(
        db, arxiv, propagated, key, f"year = {datasets.VALID_YEAR}", 10)])
    for key in scores.cal_keys
])

In [ ]:
# Deterministic split + biased subsample (seed pinned). gamma is the shift
# strength; the biased calibration set is 90% of its pool.
SEED, GAMMA = 0, 1.2
rng = np.random.default_rng(SEED)
order = rng.permutation(len(nc))
half = len(nc) // 2
test_idx, calpool_idx = order[:half], order[half:]
CAL_SIZE = int(0.9 * len(calpool_idx))

# standardize the shift feature on the cal pool
s = (shift_axis - shift_axis[calpool_idx].mean()) / (shift_axis[calpool_idx].std() + 1e-12)
# bias the cal subsample toward LOW shift-feature (easy / low-nonconformity) points:
# biased density ∝ exp(-gamma·s). The true test/cal likelihood ratio is then ∝ exp(+gamma·s).
p = np.exp(-GAMMA * s[calpool_idx])
p /= p.sum()
cal_sel = rng.choice(calpool_idx, size=CAL_SIZE, replace=False, p=p)

cal_nc = nc[cal_sel]
test_nc = nc[test_idx]
cal_s = s[cal_sel]
print(f"biased-cal mean nonconformity: {cal_nc.mean():.3f}")
print(f"holdout   mean nonconformity: {test_nc.mean():.3f}   (cal pulled lower — the shift)")

The diagnostic that distinguishes this from the keystone: on the biased calibration
set, the **nonconformity correlates clearly and positively with the shift feature**.
This is the inverse of the keystone's ≈ 0.

In [ ]:
shift_corr = float(np.corrcoef(cal_nc, cal_s)[0, 1])
print(f"corr(nonconformity, shift-feature): {shift_corr:+.3f}")
contracts.assert_close("arxiv.conformal.synthetic_shift_corr", shift_corr, tol=0.08)

Now the two passes, apples-to-apples — same APS nonconformity, the **only** difference
is the weights. The marginal pass takes the finite-sample $(1-\alpha)$ quantile of the
calibration nonconformity; the weighted pass takes the *weighted* quantile under the
Tibshirani likelihood-ratio weights $w \propto \exp(+\gamma s)$ that up-weight the
under-represented hard points (Tibshirani et al. 2019).

In [ ]:
m = len(cal_nc)
asc = np.argsort(cal_nc)

# Marginal split-conformal quantile.
k = int(np.ceil((m + 1) * (1 - alpha)))
q_marginal = cal_nc[asc][min(k, m) - 1]
marginal_cov = float(np.mean(test_nc <= q_marginal))

# Weighted split-conformal quantile (Tibshirani 2019): the LR weights ∝ exp(+gamma·s).
w = np.exp(GAMMA * cal_s)
w = w / w.sum()
cw = np.cumsum(w[asc])
q_weighted = cal_nc[asc][min(int(np.searchsorted(cw, 1 - alpha)), m - 1)]
weighted_cov = float(np.mean(test_nc <= q_weighted))

print(f"nominal coverage:            {1 - alpha:.2f}")
print(f"MARGINAL coverage:           {marginal_cov:.3f}   (under-covers)")
print(f"WEIGHTED  coverage:          {weighted_cov:.3f}   (restored to ≥ nominal)")
print(f"Δ (weighted − marginal):     {weighted_cov - marginal_cov:+.3f}")

In [ ]:
contracts.assert_close("arxiv.conformal.synthetic_marginal_coverage", marginal_cov, tol=0.03)
contracts.assert_close("arxiv.conformal.synthetic_weighted_coverage", weighted_cov, tol=0.03)
if SCALE is scale.Scale.FULL:
    # The inverse of the keystone's no-op: weighting moves coverage from clear
    # under-coverage to nominal, because the shift is score-aligned.
    assert marginal_cov < 1 - alpha
    assert weighted_cov >= 1 - alpha
    assert weighted_cov - marginal_cov > 0.05
    assert shift_corr > 0.4

Marginal split-conformal **under-covers** on the biased calibration set, and
weighted split-conformal **restores** coverage to nominal. Weighted conformal's
guarantee is in expectation over the calibration draw; at `small` scale the
biased calibration set is a few dozen papers, so its weighted quantile lands less
reliably, and the restore is asserted as the full-scale finding.

Note these are **client-local numpy** computations. The engine's `conformalize*`
surface is the **marginal** one (valid under exchangeability); Mondrian / weighted
conformal is not an OSS Python lever — the weighting here is the consumer's
client-side construction, exactly where the conformal doctrine places the cohort
choice.

The session's work is done, so it is closed. An embedded engine holds its catalog until
`close()` returns, which is why `close()` comes before anything removes the directory the
catalog lives in.

In [ ]:
db.close()

## The two halves of one thesis

> **Weighted conformal repairs a covariate shift if and only if the shift moves the
> nonconformity-score distribution.** The keystone (tier 04) is one half: on the real
> ogbn-arxiv time-split the shift is *orthogonal* to the score (a near-zero correlation for
> classification; for regression, residual magnitude is uncorrelated with
> test-likeness *within* the calibration set, even though the residuals themselves do
> shift across eras), so reweighting the calibration CDF along a direction the score
> barely depends on cannot move the quantile — a no-op, and the honest remedy is a
> governed, time-aware cohort. This chapter is the other half: a
> transparently-constructed, score-aligned
> shift, positively correlated with the score, on which marginal under-covers and the
> weighted restore (Tibshirani et al. 2019) is real. Together they are the complete statement —
> neither half alone is the theorem. A graph-aware, productionized version of the governed
> cohort is out of scope here (Huang et al. 2023); the OSS Python surface exposes the
> marginal `conformalize*` and leaves the cohort/weight choice to you.

## References

- Vovk, Vladimir, Gammerman, Alexander, Shafer, Glenn (2005) *Algorithmic Learning in a Random World* Springer.
- Angelopoulos, Anastasios N., Bates, Stephen (2021) *A Gentle Introduction to Conformal Prediction and Distribution-Free Uncertainty Quantification* arXiv preprint arXiv:2107.07511 Published in Foundations and Trends in Machine Learning, 16(4):494–591, 2023.
- Romano, Yaniv, Sesia, Matteo, Candès, Emmanuel J. (2020) *Classification with Valid and Adaptive Coverage* Advances in Neural Information Processing Systems 33 (NeurIPS).
- Angelopoulos, Anastasios N., Bates, Stephen, Malik, Jitendra, Jordan, Michael I. (2021) *Uncertainty Sets for Image Classifiers using Conformal Prediction* International Conference on Learning Representations (ICLR).
- Romano, Yaniv, Patterson, Evan, Candès, Emmanuel J. (2019) *Conformalized Quantile Regression* Advances in Neural Information Processing Systems 32 (NeurIPS).
- Tibshirani, Ryan J., Barber, Rina Foygel, Candès, Emmanuel J., Ramdas, Aaditya (2019) *Conformal Prediction Under Covariate Shift* Advances in Neural Information Processing Systems 32 (NeurIPS).
- Barber, Rina Foygel, Candès, Emmanuel J., Ramdas, Aaditya, Tibshirani, Ryan J. (2023) *Conformal Prediction Beyond Exchangeability* The Annals of Statistics.
- Huang, Kexin, Jin, Ying, Candès, Emmanuel, Leskovec, Jure (2023) *Uncertainty Quantification over Graph with Conformalized Graph Neural Networks* Advances in Neural Information Processing Systems 36 (NeurIPS).